In [17]:
import os
gpu_ids = [0]
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(map(str, gpu_ids))
import torch

In [43]:
# color_transfer_dataset.py
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image, ImageColor
from torchvision import transforms

class ColorTransferDataset(Dataset):
    def __init__(self, original_csv, fake_csv, original_image_dir, recolored_image_dir, transform_size=(256, 256), mode="both"):
        self.df_original = pd.read_csv(original_csv)
        self.df_fake = pd.read_csv(fake_csv)
        self.original_image_dir = original_image_dir
        self.recolored_image_dir = recolored_image_dir
        self.mode = mode  # "recolor", "reconstruct", or "both"
        self.transform = transforms.Compose([
            transforms.Resize(transform_size),
            transforms.ToTensor()
        ])
        self.pairs = self._prepare_pairs()

    def _prepare_pairs(self):
        pairs = []

        if self.mode in ["both", "recolor"]:
            for _, row in self.df_fake.iterrows():
                fake_id = str(row["id"])
                original_id = fake_id.split("_")[0]
                color = row["baseColour"]
                original_path = os.path.join(self.original_image_dir, f"{original_id}.jpg")
                fake_path = os.path.join(self.recolored_image_dir, f"{fake_id}.jpg")
                if os.path.exists(original_path) and os.path.exists(fake_path):
                    pairs.append((original_id, color, fake_id))

        return pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        original_id, target_color, target_id = self.pairs[idx]

        original_path = os.path.join(self.original_image_dir, f"{original_id}.jpg")
        target_path = os.path.join(self.recolored_image_dir, f"{target_id}.jpg")
        original_img = Image.open(original_path).convert("RGB")
        target_img = Image.open(target_path).convert("RGB")

        original_tensor = self.transform(original_img)
        target_tensor = self.transform(target_img)

        rgb = torch.tensor(ImageColor.getrgb(target_color)).float() / 255.
        color_patch = rgb.view(3, 1, 1).expand(3, *original_tensor.shape[1:])
        input_tensor = torch.cat([original_tensor, color_patch], dim=0)

        return input_tensor, target_tensor

In [44]:
# model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class ColorConditionedUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = UNetBlock(6, 64)
        self.enc2 = UNetBlock(64, 128)
        self.enc3 = UNetBlock(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.up3 = UNetBlock(256, 128)
        self.up2 = UNetBlock(128, 64)
        self.final = nn.Conv2d(64, 3, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        d3 = self.up3(F.interpolate(e3, scale_factor=2, mode='bilinear', align_corners=False))
        d2 = self.up2(F.interpolate(d3, scale_factor=2, mode='bilinear', align_corners=False))
        return self.final(d2)

In [45]:
# train.py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
def train_model(
    original_csv,
    fake_csv,
    original_image_dir,
    recolored_image_dir,
    batch_size=8,
    lr=1e-4,
    epochs=20,
    save_path="color_unet.pth",
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("✅ Using device:", device)

    dataset = ColorTransferDataset(original_csv, fake_csv, original_image_dir, recolored_image_dir, mode="both")
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

    model = ColorConditionedUNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")

        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item())

        print(f"✅ Epoch {epoch+1} - Avg Loss: {total_loss / len(dataloader):.4f}")

    torch.save(model.state_dict(), save_path)
    print(f"✅ Model saved to {save_path}")

In [50]:
train_model(
    original_csv="./pure_tshirt.csv",
    fake_csv="./metadata_augmented_lab.csv",
    original_image_dir = "../fashion-dataset/images",
    recolored_image_dir="../fashion-dataset/fake_images",
    batch_size=8,
    epochs=100,
    save_path="0404_1.pth"
)


✅ Using device: cuda


Epoch 1/100:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1/100: 100%|██████████| 10/10 [00:02<00:00,  3.98it/s, loss=0.662]


✅ Epoch 1 - Avg Loss: 0.7607


Epoch 2/100: 100%|██████████| 10/10 [00:02<00:00,  3.92it/s, loss=0.217]


...

✅ Epoch 98 - Avg Loss: 0.0531


Epoch 99/100: 100%|██████████| 10/10 [00:02<00:00,  3.85it/s, loss=0.0284]


✅ Epoch 99 - Avg Loss: 0.0484


Epoch 100/100: 100%|██████████| 10/10 [00:02<00:00,  3.75it/s, loss=0.0565]

✅ Epoch 100 - Avg Loss: 0.0484
✅ Model saved to 0404_1.pth


In [15]:
import os
import pandas as pd
import numpy as np
from PIL import Image, ImageColor
import cv2

def recolor_multiple_examples_lab(
    csv_path,
    image_dir,
    mask_dir,
    save_dir,
    target_colors=("Red", "Green", "Yellow"),
    output_csv_path="metadata_augmented_lab.csv",
    max_images=50  # 👈 number of images to recolor
):
    df = pd.read_csv(csv_path)
    rows = df.iloc[:max_images]  # select first N images

    os.makedirs(save_dir, exist_ok=True)
    new_rows = []

    for _, row in rows.iterrows():
        image_id = str(row["id"])
        image_path = os.path.join(image_dir, f"{image_id}.jpg")
        mask_path = os.path.join(mask_dir, f"{image_id}.npy")

        if not os.path.exists(image_path):
            print(f"❌ Missing image for {image_id}")
            continue
        if not os.path.exists(mask_path):
            print(f"❌ Missing mask for {image_id}")
            continue

        # Load image and convert to LAB
        image_bgr = cv2.imread(image_path)
        image_lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
        H, W = image_lab.shape[:2]

        # Load and resize mask
        mask_np = np.load(mask_path)
        mask_resized = cv2.resize(mask_np, (W, H))
        mask_binary = (mask_resized > 0.5).astype(np.uint8)
        mask_binary = 1 - mask_binary  # Invert if needed

        for color in target_colors:
            if color.lower() == row["baseColour"].lower():
                continue

            try:
                target_rgb = np.array(ImageColor.getrgb(color)).astype(np.uint8).reshape(1, 1, 3)
                target_lab = cv2.cvtColor(target_rgb, cv2.COLOR_RGB2LAB)[0, 0]
            except:
                print(f"Invalid color: {color}")
                continue

            # Recolor using LAB blending
            recolored_lab = image_lab.copy()
            recolored_lab[:, :, 1] = np.where(mask_binary, target_lab[1], recolored_lab[:, :, 1])
            recolored_lab[:, :, 2] = np.where(mask_binary, target_lab[2], recolored_lab[:, :, 2])

            recolored_bgr = cv2.cvtColor(recolored_lab, cv2.COLOR_LAB2BGR)
            recolored_rgb = cv2.cvtColor(recolored_bgr, cv2.COLOR_BGR2RGB)

            new_id = f"{image_id}_{color.lower()}"
            new_path = os.path.join(save_dir, f"{new_id}.jpg")
            Image.fromarray(recolored_rgb).save(new_path)

            new_row = row.copy()
            new_row["id"] = new_id
            new_row["baseColour"] = color
            new_rows.append(new_row)

            print(f"✅ Saved recolored image: {new_path}")

    # Save CSV
    if new_rows:
        pd.DataFrame(new_rows).to_csv(output_csv_path, index=False)
        print(f"✅ CSV with {len(new_rows)} recolored entries saved to: {output_csv_path}")
    else:
        print("⚠️ No new recolored rows were generated.")


In [16]:
recolor_multiple_examples_lab(
    csv_path="./pure_tshirt.csv",
    image_dir="../fashion-dataset/images",
    mask_dir="./cached_masks",
    save_dir="../fashion-dataset/fake_images",  # or use "recolored" if you want to separate
    target_colors=["Red", "Green"]
)


✅ Saved recolored image: ../fashion-dataset/fake_images/4729_red.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/38630_red.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/38630_green.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/2288_red.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/2288_green.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/8322_red.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/8322_green.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/30650_red.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/30650_green.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/7503_red.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/7503_green.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/8325_red.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/8325_green.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/38637_green.jpg
✅ Saved recolor

===============

In [51]:
import torch
from torchvision import transforms
from PIL import Image, ImageColor
import os

def recolor_image(model_path, image_path, base_color, output_path, image_size=(256, 256)):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load and prepare model
    model = ColorConditionedUNet().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Preprocess image
    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor()
    ])

    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image)

    # Create color patch
    rgb = torch.tensor(ImageColor.getrgb(base_color)).float() / 255.
    color_patch = rgb.view(3, 1, 1).expand(3, *image_tensor.shape[1:])
    input_tensor = torch.cat([image_tensor, color_patch], dim=0).unsqueeze(0).to(device)  # [1, 6, H, W]

    # Predict
    with torch.no_grad():
        output = model(input_tensor).squeeze(0).cpu()

    # Convert output to image
    output_image = transforms.ToPILImage()(output.clamp(0, 1))
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    output_image.save(output_path)
    print(f"✅ Recolored image saved to {output_path}")


In [55]:
recolor_image(
    model_path="0404_1.pth",
    image_path="../fashion-dataset/images/36199.jpg",
    base_color="Green",  # or "Red", "Black", etc.
    output_path="./recolored/36199_green_pred.jpg"
)


✅ Recolored image saved to ./recolored/36199_green_pred.jpg


/tmp/ipykernel_2429658/613027857.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))
